# Hanogt AI — Kaggle'da ücretsiz eğitim

Bu defter Hanogt AI'ın dil modelini (**Qwen3-8B**, QLoRA) Kaggle'ın ücretsiz GPU'suyla eğitir.
Veri seti ve sonuçlar senin **özel** Hugging Face depolarında durur; defter kesilirse bir sonraki
çalıştırmada kaldığı yerden devam eder.

### Bir kez yapılacaklar
1. **Telefon doğrulaması:** kaggle.com → profil resmi → *Settings* → *Phone Verification*.
   Bu olmadan GPU ve internet açılmaz.
2. **Oturum ayarları** (sağ panel → *Session options*):
   - *Accelerator*: **GPU T4 x2**
   - *Internet*: **On**
3. **Token:** üst menü → *Add-ons* → *Secrets* → *Add Secret*:
   - Label: `HF_TOKEN`
   - Value: huggingface.co → Settings → Access Tokens'tan aldığın **Write** token
   - Kaydet ve bu deftere bağlı olduğunu gösteren kutuyu işaretle.
   - Token'ı asla bir hücreye yazma.

### Çalıştırma
Üstte **Save Version → Save & Run All (Commit)**. Tarayıcıyı kapatabilirsin.
Kaggle bir oturuma en fazla 12 saat verir; defter 11 saatte kendisi kaydedip durur.
Eğitim bitmediyse aynı adımı tekrarla: son ara kayıttan devam eder.
Haftalık ücretsiz GPU kotası yaklaşık 30 saattir.

### Gerekenler
Veri seti önce özel depoya yüklenmiş olmalı (`HanStudios/hanogt-sft-v2`).
Bunu Claude Code ortamından `python3 training/hf_job.py upload --dataset-repo HanStudios/hanogt-sft-v2` yapar.

In [ ]:
# Ayarlar: depolar senin hesabında ve özel.
DATA_REPO = "HanStudios/hanogt-sft-v2"            # veri seti ve eğitim kodu
MODEL_REPO = "HanStudios/hanogt-ai-qwen3-8b-lora"  # ara kayıtlar ve eğitilen bağdaştırıcı
BASE_MODEL = "Qwen/Qwen3-8B"
EPOCHS = 1
MAX_SEQ_LEN = 4096      # T4'ün belleğine göre; daha uzun örnekler atlanır
HOURS = 11.0            # Kaggle oturumu 12 saat: kayıt ve yükleme için 1 saat pay
USE_BOTH_GPUS = True    # GPU T4 x2: iki kart birlikte çalışır (yaklaşık iki kat hız)

In [ ]:
import os
from kaggle_secrets import UserSecretsClient

os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
print("HF_TOKEN hazır (değeri yazdırılmaz)")

In [ ]:
!pip install -q -U "transformers>=4.56" "peft>=0.17" "accelerate>=1.2" "bitsandbytes>=0.45" "huggingface_hub>=0.35" "safetensors>=0.4"

In [ ]:
import torch

assert torch.cuda.is_available(), "GPU yok: sağ panel → Session options → Accelerator → GPU T4 x2"
gpus = torch.cuda.device_count()
print(f"{gpus} GPU:", [torch.cuda.get_device_name(index) for index in range(gpus)])
print("bf16:", torch.cuda.is_bf16_supported(), "(T4 ve P100'de fp16 kullanılır)")

In [ ]:
import json
from huggingface_hub import snapshot_download

work = snapshot_download(DATA_REPO, repo_type="dataset", local_dir="/kaggle/working/data", token=os.environ["HF_TOKEN"])
manifest = json.load(open(f"{work}/manifest.json", encoding="utf-8"))
print(json.dumps({key: manifest[key] for key in ("train", "eval") if key in manifest}, indent=2, ensure_ascii=False)[:1500])

In [ ]:
import shlex
import subprocess
import sys

launcher = ["torchrun", f"--nproc_per_node={gpus}"] if USE_BOTH_GPUS and gpus > 1 else [sys.executable]
command = launcher + [
    f"{work}/code/train_lora.py",
    "--preset", "small",
    "--base-model", BASE_MODEL,
    "--data", f"{work}/train.jsonl",
    "--eval-data", f"{work}/eval.jsonl",
    "--output", "/kaggle/working/out",
    "--epochs", str(EPOCHS),
    "--max-seq-len", str(MAX_SEQ_LEN),
    "--hub-checkpoints", MODEL_REPO,
    "--resume-from-hub",
    "--time-budget-hours", str(HOURS),
    "--push-to-hub", MODEL_REPO,
]
print(" ".join(shlex.quote(part) for part in command))
subprocess.run(command, check=True)

### Sonra
- **Eğitim bittiyse** bağdaştırıcı `MODEL_REPO`'da durur (`adapter_model.safetensors` ve `hanogt-training.json`).
  Claude Code oturumunda "Kaggle eğitimi bitti" de; değerlendirmeyi (code-bench, Türkçe sorular) ve
  yayına hazırlığı ben yaparım.
- **Süre dolduysa** çıktıda "time budget … reached" yazar.
  Yeniden *Save & Run All* yap: `last-checkpoint`'ten devam eder.